In [ ]:
import os, time, glob
import numpy as np, pandas as pd, cv2, torch
from torch.utils.data import Dataset, DataLoader
from transformers import ViTModel

DEBUG = True            # True = test on 50 random videos. Set False for the full run.
N_FRAMES = 8
SHARD_SIZE = 1000
BATCH_VIDEOS = 16       # 16 videos x 8 frames = 128 images per batch
NUM_WORKERS = 4
OUT_DIR = "/kaggle/working/vit_feats" + ("_debug" if DEBUG else "")
os.makedirs(OUT_DIR, exist_ok=True)

# find master.csv wherever Kaggle mounted it
MASTER = glob.glob("/kaggle/input/**/master.csv", recursive=True)[0]
print("master:", MASTER)

df = pd.read_csv(MASTER)
if DEBUG:
    df = df.sample(50, random_state=0).reset_index(drop=True)
print(len(df), "videos")

In [ ]:
def load_frames(path, n=N_FRAMES, size=224):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    want = set(np.linspace(0, total - 1, n).astype(int).tolist())
    frames, i = [], 0
    while len(frames) < n:
        ok = cap.grab()
        if not ok:
            break
        if i in want:
            ok, f = cap.retrieve()
            if ok:
                h, w = f.shape[:2]
                s = min(h, w)                               # center square crop
                y0, x0 = (h - s) // 2, (w - s) // 2
                f = f[y0:y0 + s, x0:x0 + s]
                f = cv2.resize(f, (size, size), interpolation=cv2.INTER_AREA)
                frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        i += 1
    cap.release()
    failed = len(frames) == 0
    if failed:
        frames = [np.zeros((size, size, 3), np.uint8)] * n
    while len(frames) < n:                                  # pad if a frame was missed
        frames.append(frames[-1])
    return np.stack(frames[:n]), failed

class VideoDS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x, failed = load_frames(self.paths[i])
        return torch.from_numpy(x), failed                  # uint8 [8,224,224,3]

In [ ]:
device = "cuda"
vit = ViTModel.from_pretrained("google/vit-base-patch16-224", add_pooling_layer=False)
vit = vit.to(device).half().eval()
mean = torch.tensor([0.5, 0.5, 0.5], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.5, 0.5, 0.5], device=device).view(1, 3, 1, 1)

@torch.no_grad()
def embed(batch):                                           # batch: uint8 [B,8,224,224,3]
    B = batch.shape[0]
    x = batch.to(device).view(B * N_FRAMES, 224, 224, 3).permute(0, 3, 1, 2).float() / 255.0
    x = ((x - mean) / std).half()
    cls = vit(pixel_values=x).last_hidden_state[:, 0]       # [B*8, 768]
    return cls.view(B, N_FRAMES, 768).float().cpu().numpy().astype(np.float16)

In [ ]:
n_shards = (len(df) + SHARD_SIZE - 1) // SHARD_SIZE
t0, done_videos, n_failed = time.time(), 0, 0

for s in range(n_shards):
    out_f = f"{OUT_DIR}/shard_{s:04d}.npy"
    fail_f = f"{OUT_DIR}/fail_{s:04d}.npy"
    if os.path.exists(out_f) and os.path.exists(fail_f):
        print("skip", s); continue

    sub = df.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]
    dl = DataLoader(VideoDS(sub["path"].tolist()), batch_size=BATCH_VIDEOS,
                    num_workers=NUM_WORKERS, shuffle=False)
    feats, fails = [], []
    for batch, failed in dl:
        feats.append(embed(batch)); fails.append(failed.numpy())
    feats, fails = np.concatenate(feats), np.concatenate(fails)

    np.save(out_f, feats); np.save(fail_f, fails)
    done_videos += len(sub); n_failed += int(fails.sum())
    el = time.time() - t0
    print(f"shard {s+1}/{n_shards} | {feats.shape} | {el/done_videos:.2f}s/video | failed so far: {n_failed}")

In [ ]:
if not DEBUG:
    feats = np.concatenate([np.load(f) for f in sorted(glob.glob(f"{OUT_DIR}/shard_*.npy"))])
    fails = np.concatenate([np.load(f) for f in sorted(glob.glob(f"{OUT_DIR}/fail_*.npy"))])
    assert len(feats) == len(df)
    np.save(f"{OUT_DIR}/vit_feats_all.npy", feats)
    np.save(f"{OUT_DIR}/vit_fail_all.npy", fails)
    print(feats.shape, feats.dtype, "failed videos:", fails.sum())